# 🚗 Entrenamiento YOLO11n para Reconocimiento de Placas (ALPR)

Este notebook está diseñado para ejecutarse en **Google Colab** aprovechando su GPU gratuita.

➡️ Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

**Flujo completo:**
1. Instalar dependencias
2. Montar Google Drive (para guardar el modelo entrenado)
3. Descargar dataset (Roboflow)
4. Entrenar YOLO11 Nano
5. Validar métricas
6. Exportar a `.onnx` y copiar a Drive

## 1. Instalación de Dependencias

In [ ]:
!pip install ultralytics roboflow --quiet
import ultralytics
ultralytics.checks()
print('✅ Entorno listo')

## 2. Montar Google Drive
Aquí guardaremos el modelo `best.onnx` al final para que no se pierda cuando Colab se desconecte.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Carpeta de destino en tu Drive — cámbiala si quieres
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ALPR_Model'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'✅ Drive montado. Los modelos se guardarán en: {DRIVE_OUTPUT_DIR}')

## 3. Descarga del Dataset (Roboflow)

1. Entra a [roboflow.com](https://app.roboflow.com), crea tu proyecto de detección de placas.
2. En **Export Dataset** elige el formato **YOLOv8** (compatible con YOLO11).
3. Copia el snippet de código que te da Roboflow y pégalo abajo.

In [ ]:
# ⚠️  Reemplaza este bloque con el código que te da Roboflow al exportar.
# El formato es YOLOv8. Ejemplo:
from roboflow import Roboflow
rf = Roboflow(api_key="TU_API_KEY_AQUI")
project = rf.workspace("tu-workspace").project("tu-proyecto-alpr")
version = project.version(1)
dataset = version.download("yolov8")
DATASET_YAML = dataset.location + "/data.yaml"
print(f'✅ Dataset descargado. YAML: {DATASET_YAML}')

## 4. Entrenamiento del Modelo

Usamos `yolo11n.pt` (Nano) — el modelo más ligero para maximizar FPS en CPU (Matebook).
- **epochs=100**: Buen punto de partida. Aumenta si ves que las métricas siguen mejorando.
- **imgsz=640**: Tamaño estándar, no cambiar.
- **patience=20**: Para early stopping si no mejora en 20 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    patience=20,       # Early stopping
    batch=16,          # Ajustar según la VRAM disponible
    device=0,          # GPU 0 (T4 en Colab)
    project='/content/runs',
    name='alpr_v1',
    exist_ok=True
)

BEST_PT = '/content/runs/alpr_v1/weights/best.pt'
print(f'✅ Entrenamiento finalizado. Mejor modelo: {BEST_PT}')

## 5. Validación y Métricas
Revisamos el desempeño del modelo entrenado sobre el conjunto de validación.

In [ ]:
import os
from IPython.display import Image, display

# Validar el modelo
model_best = YOLO(BEST_PT)
metrics = model_best.val()
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

# Mostrar tarjetas visuales
results_path = '/content/runs/alpr_v1'
for img_name in ['confusion_matrix.png', 'results.png', 'PR_curve.png']:
    img_path = os.path.join(results_path, img_name)
    if os.path.exists(img_path):
        print(f'--- {img_name} ---')
        display(Image(filename=img_path, width=700))

## 6. Exportación a ONNX + Copia a Google Drive

Exportamos `best.pt` → `best.onnx`. Luego:
1. Lo copiamos a tu Google Drive.
2. Descárgalo desde Drive a tu máquina local.
3. Ponlo en la carpeta `models/` del proyecto → `models/best.onnx`.
4. En el `.env` local: `DETECTION_MODEL=best.onnx`

In [ ]:
import shutil

print('Exportando a ONNX...')
onnx_path = model_best.export(format='onnx', imgsz=640, dynamic=False)
print(f'✅ ONNX guardado en: {onnx_path}')

# Copiar a Google Drive
drive_onnx = os.path.join(DRIVE_OUTPUT_DIR, 'best.onnx')
drive_pt   = os.path.join(DRIVE_OUTPUT_DIR, 'best.pt')
shutil.copy(str(onnx_path), drive_onnx)
shutil.copy(BEST_PT, drive_pt)
print(f'✅ Modelo copiado a Drive:')
print(f'   ONNX → {drive_onnx}')
print(f'   PT   → {drive_pt}')
print()
print('📋 Próximos pasos:')
print('  1. Descarga best.onnx desde Google Drive')
print('  2. Colócalo en ProyectoPlacasV1/models/best.onnx')
print('  3. En tu .env local agrega: DETECTION_MODEL=best.onnx')
print('  4. Corre: python src/app.py')